In [3]:
# Imports

import plotly.express as px
import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from umap import UMAP

c:\Users\Lukas\Master_AI\3rd_Semester\UE_AI_and_Visualization\UE\Project\.venv-irony\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Define constants for the model, dataset, batch size, and label names

MODEL_ID = "cardiffnlp/twitter-roberta-base-irony"
DATASET_ID = "cardiffnlp/tweet_eval"
BATCH_SIZE = 16
LABEL_NAMES = {0: "Non-ironic", 1: "Ironic"}

In [5]:
# Preprocess tweets by normalizing usernames and URLs

def preprocess_tweet(text: str) -> str:
    """Normalize usernames and URLs as described in the model card."""
    normalized_tokens = []

    for token in text.split(" "):
        if token.startswith("@") and len(token) > 1:
            token = "@user"
        elif token.startswith("http"):
            token = "http"
        normalized_tokens.append(token)

    return " ".join(normalized_tokens)

In [6]:
# Load the held-out irony split and keep the original text for hover labels

dataset = load_dataset(DATASET_ID, "irony", split="test")
tweets = dataset.to_pandas()
tweets["Reference label"] = tweets["label"].map(LABEL_NAMES)
texts = [preprocess_tweet(text) for text in tweets["text"]]

# Load the tokenizer and put the classifier in evaluation mode

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.eval()

embedding_batches = []
probability_batches = []

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2165.39it/s]


In [7]:
# Run inference in batches to limit memory usage

with torch.inference_mode():
    for start in range(0, len(texts), BATCH_SIZE):
        batch = tokenizer(
            texts[start : start + BATCH_SIZE],
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        outputs = model(**batch, output_hidden_states=True)

        # Use the final-layer representation of the first token for each tweet.
        # This is the representation passed to RoBERTa's classification head.
        embedding_batches.append(outputs.hidden_states[-1][:, 0, :])

        # Convert classification logits into probabilities for both classes.
        probability_batches.append(outputs.logits.softmax(dim=-1))

In [8]:
# Combine the batches in the original dataset order

X = torch.cat(embedding_batches).numpy()
probabilities = torch.cat(probability_batches)
predicted_ids = probabilities.argmax(dim=1).tolist()

tweets["Prediction"] = [LABEL_NAMES[label] for label in predicted_ids]
tweets["Irony score"] = probabilities[:, 1].numpy()
tweets["Classification"] = (
    tweets["Prediction"]
    .eq(tweets["Reference label"])
    .map({True: "Correct", False: "Misclassified"})
)

In [9]:
# Fit UMAP on the embeddings. Labels are used later for display.
projection = UMAP(
    n_components=2,
    n_neighbors=15,
    metric="cosine",
    random_state=42,
)
tweets[["UMAP 1", "UMAP 2"]] = projection.fit_transform(X)

# Color points by the reference label and mark classification errors with an X.
fig = px.scatter(
    tweets,
    x="UMAP 1",
    y="UMAP 2",
    color="Reference label",
    symbol="Classification",
    symbol_map={"Correct": "circle", "Misclassified": "x"},
    hover_data={
        "text": True,
        "Prediction": True,
        "Irony score": ":.3f",
    },
    title="TweetEval irony test set · RoBERTa embeddings",
)

# Save an interactive plot that can be opened in a browser.
fig.write_html("ironie_umap.html")

c:\Users\Lukas\Master_AI\3rd_Semester\UE_AI_and_Visualization\UE\Project\.venv-irony\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
